# Explore patient 11-37493 data
Scratch exploration ahead of writing the final `scripts/closest_anatomy.py` script.

In [ ]:
import glob
import pydicom

PATIENT_DIR = '../data/11-37493'
ct_dir = f'{PATIENT_DIR}/11-37493_11-37493_CT_2011-12-15_112426_._CT.WB.50cm_n263__00000'
pt_dir = f'{PATIENT_DIR}/11-37493_11-37493_PT_2011-12-15_112426_._PET.  HD.AC_n263__00000'
rt_dir = f'{PATIENT_DIR}/11-37493_11-37493_RTst_2011-12-15_112426_._Lesions_n1__00000'

ct_files = sorted(glob.glob(f'{ct_dir}/*.dcm'))
pt_files = sorted(glob.glob(f'{pt_dir}/*.dcm'))
rt_files = sorted(glob.glob(f'{rt_dir}/*.dcm'))
len(ct_files), len(pt_files), len(rt_files)

(263, 263, 1)

In [2]:
ct = pydicom.dcmread(ct_files[0], stop_before_pixels=True)
pt = pydicom.dcmread(pt_files[0], stop_before_pixels=True)
print('CT SeriesInstanceUID:', ct.SeriesInstanceUID)
print('PT SeriesInstanceUID:', pt.SeriesInstanceUID)
print('CT rows/cols/spacing/thickness:', ct.Rows, ct.Columns, ct.PixelSpacing, ct.SliceThickness)
print('PT rows/cols/spacing/thickness:', pt.Rows, pt.Columns, pt.PixelSpacing, pt.SliceThickness)

CT SeriesInstanceUID: 1.2.840.113654.2.70.1.12188168587231107189442657105736749486
PT SeriesInstanceUID: 1.2.840.113654.2.70.1.7567817886330450357650383961264298479
CT rows/cols/spacing/thickness: 512 512 [0.976562, 0.976562] 3.750000
PT rows/cols/spacing/thickness: 192 192 [3.6458332538605, 3.6458332538605] 3.2700


In [3]:
ds = pydicom.dcmread(rt_files[0])
print('Modality:', ds.Modality)
print('StructureSetLabel:', getattr(ds, 'StructureSetLabel', None))
print()
rois = [(roi.ROINumber, roi.ROIName) for roi in ds.StructureSetROISequence]
for num, name in rois:
    print(num, name)
print()
for frs in ds.ReferencedFrameOfReferenceSequence:
    print('FrameOfReferenceUID:', frs.FrameOfReferenceUID)
    for study in frs.RTReferencedStudySequence:
        for series in study.RTReferencedSeriesSequence:
            print('Referenced SeriesInstanceUID:', series.SeriesInstanceUID)
            print('Num referenced images:', len(series.ContourImageSequence))

Modality: RTSTRUCT
StructureSetLabel: RTstruct

1 ROI-1
2 ROI-2
3 ROI-3
4 ROI-4
5 ROI-5
6 ROI-6
7 ROI-7
8 ROI-8
9 ROI-9
10 ROI-10
11 ROI-11
12 ROI-12
13 ROI-13
14 ROI-14
15 ROI-15
16 ROI-16
17 ROI-17
18 ROI-18
19 ROI-19
20 ROI-20
21 ROI-21
22 ROI-22
23 ROI-23
24 ROI-24
25 ROI-25
26 ROI-26
27 ROI-27
28 ROI-28
29 ROI-29
30 ROI-30
31 ROI-31
32 ROI-32
33 ROI-33
34 ROI-34
35 SUV Peak Sphere 19
36 SUV Peak Sphere 11
37 SUV Peak Sphere 24
38 SUV Peak Sphere 25
39 SUV Peak Sphere 1
40 SUV Peak Sphere 5
41 SUV Peak Sphere 7
42 SUV Peak Sphere 29
43 SUV Peak Sphere 23
44 SUV Peak Sphere 32
45 SUV Peak Sphere 6
46 SUV Peak Sphere 31
47 SUV Peak Sphere 10
48 SUV Peak Sphere 15
49 SUV Peak Sphere 22
50 SUV Peak Sphere 21
51 SUV Peak Sphere 2
52 SUV Peak Sphere 14
53 SUV Peak Sphere 16
54 SUV Peak Sphere 12
55 SUV Peak Sphere 20
56 SUV Peak Sphere 3
57 SUV Peak Sphere 4
58 SUV Peak Sphere 17
59 SUV Peak Sphere 33
60 SUV Peak Sphere 9
61 SUV Peak Sphere 30
62 SUV Peak Sphere 18
63 SUV Peak Sphere 8
6

In [4]:
referenced_sop_uids = set()
for frs in ds.ReferencedFrameOfReferenceSequence:
    for study in frs.RTReferencedStudySequence:
        for series in study.RTReferencedSeriesSequence:
            for contour_image in series.ContourImageSequence:
                referenced_sop_uids.add(contour_image.ReferencedSOPInstanceUID)

ct_sop_uids = set()
ct_sop_to_file = {}
for f in ct_files:
    d = pydicom.dcmread(f, stop_before_pixels=True)
    ct_sop_uids.add(d.SOPInstanceUID)
    ct_sop_to_file[d.SOPInstanceUID] = f

print('Referenced by RTSTRUCT:', len(referenced_sop_uids))
print('Present in ct_dir:', len(ct_sop_uids))
print('Missing from ct_dir:', len(referenced_sop_uids - ct_sop_uids))
print('Extra in ct_dir (not referenced):', len(ct_sop_uids - referenced_sop_uids))
missing = list(referenced_sop_uids - ct_sop_uids)[:5]
missing

Referenced by RTSTRUCT: 263
Present in ct_dir: 263
Missing from ct_dir: 263
Extra in ct_dir (not referenced): 263


['1.2.840.113654.2.70.1.146298449627387866212517459178691683566',
 '1.2.840.113654.2.70.1.108096419514171783757812840498454537058',
 '1.2.840.113654.2.70.1.198771774153681517701664065408025850695',
 '1.2.840.113654.2.70.1.60305805495907694650554450723622659245',
 '1.2.840.113654.2.70.1.21281491156374268243225050401795820282']

In [5]:
# the RTSTRUCT actually references the PET series UID, not CT. Confirm frame of reference alignment.
ct_for = ct.FrameOfReferenceUID
pt_for = pt.FrameOfReferenceUID
rt_for = ds.ReferencedFrameOfReferenceSequence[0].FrameOfReferenceUID
print('CT FrameOfReferenceUID:', ct_for)
print('PT FrameOfReferenceUID:', pt_for)
print('RTSTRUCT FrameOfReferenceUID:', rt_for)
print('CT matches RT:', ct_for == rt_for)
print('PT matches RT:', pt_for == rt_for)

CT FrameOfReferenceUID: 1.2.840.113654.2.70.1.282395789882641072923715466415562956780
PT FrameOfReferenceUID: 1.2.840.113654.2.70.1.282395789882641072923715466415562956780
RTSTRUCT FrameOfReferenceUID: 1.2.840.113654.2.70.1.282395789882641072923715466415562956780
CT matches RT: True
PT matches RT: True


In [6]:
lesion_rois = [n for n in rois if not n[1].startswith('SUV Peak Sphere')]
sphere_rois = [n for n in rois if n[1].startswith('SUV Peak Sphere')]
print('lesion ROIs:', len(lesion_rois))
print('SUV peak sphere ROIs:', len(sphere_rois))
lesion_rois

lesion ROIs: 34
SUV peak sphere ROIs: 34


[('1', 'ROI-1'),
 ('2', 'ROI-2'),
 ('3', 'ROI-3'),
 ('4', 'ROI-4'),
 ('5', 'ROI-5'),
 ('6', 'ROI-6'),
 ('7', 'ROI-7'),
 ('8', 'ROI-8'),
 ('9', 'ROI-9'),
 ('10', 'ROI-10'),
 ('11', 'ROI-11'),
 ('12', 'ROI-12'),
 ('13', 'ROI-13'),
 ('14', 'ROI-14'),
 ('15', 'ROI-15'),
 ('16', 'ROI-16'),
 ('17', 'ROI-17'),
 ('18', 'ROI-18'),
 ('19', 'ROI-19'),
 ('20', 'ROI-20'),
 ('21', 'ROI-21'),
 ('22', 'ROI-22'),
 ('23', 'ROI-23'),
 ('24', 'ROI-24'),
 ('25', 'ROI-25'),
 ('26', 'ROI-26'),
 ('27', 'ROI-27'),
 ('28', 'ROI-28'),
 ('29', 'ROI-29'),
 ('30', 'ROI-30'),
 ('31', 'ROI-31'),
 ('32', 'ROI-32'),
 ('33', 'ROI-33'),
 ('34', 'ROI-34')]

In [7]:
from rt_utils import RTStructBuilder

rtstruct = RTStructBuilder.create_from(dicom_series_path=pt_dir, rt_struct_path=rt_files[0])
names = rtstruct.get_roi_names()
len(names), names[:5]

(68, ['ROI-1', 'ROI-2', 'ROI-3', 'ROI-4', 'ROI-5'])

In [8]:
import numpy as np
mask = rtstruct.get_roi_mask_by_name('ROI-1')
mask.shape, mask.dtype, mask.sum()

((192, 192, 263), dtype('bool'), np.int64(76))

In [9]:
import os
import SimpleITK as sitk

out_dir = '../outputs/11-37493'
os.makedirs(out_dir, exist_ok=True)

reader = sitk.ImageSeriesReader()
dicom_names = reader.GetGDCMSeriesFileNames(ct_dir)
reader.SetFileNames(dicom_names)
ct_image = reader.Execute()
print('CT nifti size:', ct_image.GetSize())
print('CT nifti spacing:', ct_image.GetSpacing())
print('CT nifti origin:', ct_image.GetOrigin())
print('CT nifti direction:', ct_image.GetDirection())

ct_nifti_path = f'{out_dir}/ct.nii.gz'
sitk.WriteImage(ct_image, ct_nifti_path)
ct_nifti_path

CT nifti size: (512, 512, 263)
CT nifti spacing: (0.976562, 0.976562, 3.27)
CT nifti origin: (-250.0, -250.0, -871.99)
CT nifti direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)


'../outputs/11-37493/ct.nii.gz'

In [10]:
# Build lesion mask (PET voxel space) -> patient physical (mm) -> CT voxel index pipeline
from rt_utils import image_helper

lesion_name = 'ROI-1'
mask = rtstruct.get_roi_mask_by_name(lesion_name)  # shape (Columns, Rows, num_slices) in PET voxel space
pet_voxel_idx = np.argwhere(mask)  # rows: (col, row, slice) per rt_utils convention
print('lesion voxel count:', len(pet_voxel_idx))

pixel_to_patient = image_helper.get_pixel_to_patient_transformation_matrix(rtstruct.series_data)
physical_pts = image_helper.apply_transformation_to_3d_points(pet_voxel_idx.astype(float), pixel_to_patient)
print('physical bounds (mm):')
print(' x:', physical_pts[:, 0].min(), physical_pts[:, 0].max())
print(' y:', physical_pts[:, 1].min(), physical_pts[:, 1].max())
print(' z:', physical_pts[:, 2].min(), physical_pts[:, 2].max())
centroid_physical = physical_pts.mean(axis=0)
centroid_physical

lesion voxel count: 76
physical bounds (mm):
 x: 1.8228988647460938 23.697898387908936
 y: -147.65626454353333 -129.42709827423096
 z: -796.7799906730652 -790.2399907112122


array([  13.384028  , -138.97342482, -793.38091175])

In [11]:
# Map physical point into CT voxel index space using the CT nifti we wrote via SimpleITK
ct_sitk = sitk.ReadImage(ct_nifti_path)
ct_index_continuous = ct_sitk.TransformPhysicalPointToContinuousIndex(centroid_physical.tolist())
ct_index_round = tuple(int(round(v)) for v in ct_index_continuous)
print('CT continuous index:', ct_index_continuous)
print('CT rounded index (x,y,z / col,row,slice):', ct_index_round)
print('CT size:', ct_sitk.GetSize())

# sanity: does it fall within CT bounds?
size = ct_sitk.GetSize()
in_bounds = all(0 <= ct_index_round[d] < size[d] for d in range(3))
print('in bounds:', in_bounds)

CT continuous index: (269.7053763627455, 113.6912684970986, 24.039473684210538)
CT rounded index (x,y,z / col,row,slice): (270, 114, 24)
CT size: (512, 512, 263)
in bounds: True
